# SQL query from table names - Continued

In [29]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [30]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [31]:
#context = [ {'role':'system', 'content':"""
# CREATE SEVERAL (3+) TABLES HERE
#"""} ]

context = [ {'role':'system', 'content':"""
CREATE TABLE employees (
    ID_usr int primary key,
    name varchar(100)
);
-- Sample rows:
-- (1, 'John Lopez'), (2, 'Maria Garcia'), (3, 'David Schoon')
CREATE TABLE salary (
    ID_usr int,
    year date,
    salary float
);
-- Sample rows:
-- (1, '2023-01-01', 55000), (2, '2023-01-01', 62000), (3, '2023-01-01', 48000)
CREATE TABLE studies (
    ID int,
    ID_usr int,
    educational_level int,
    Institution varchar(100),
    Years date,
    Speciality varchar(100)
);
-- Sample rows:
-- (1, 1, 3, 'MIT', '2015-09-01', 'Computer Science')
"""}]

In [32]:
#FEW SHOT SAMPLES
#context.append( {'role':'system', 'content':"""
# -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
# WRITE IN YOUR CONTEXT QUERIES HERE
# """
#})

context.append( {'role':'system', 'content':"""
-- Maintain SQL simple and efficient using valid SQLite. Answer questions for tables above.
-- Example 1:
-- Question: Who has the highest salary?
-- SQL: SELECT e.name FROM employees e JOIN salary s ON e.ID_usr = s.ID_usr ORDER BY s.salary DESC LIMIT 1;
-- Example 2:
-- Question: Which institution has the highest average salary?
-- SQL: SELECT st.Institution, AVG(sa.salary) FROM studies st JOIN employees e ON st.ID_usr=e.ID_usr JOIN salary sa ON e.ID_usr=sa.ID_usr GROUP BY st.Institution ORDER BY AVG(sa.salary) DESC LIMIT 1;
-- Example 3 (COUNT + WHERE):
-- Question: How many employees earn over 50000?
-- SQL: SELECT COUNT(*) FROM salary WHERE salary > 50000;
"""})


In [33]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [34]:
#new
context_user = context.copy()
print(return_CCRMSQL("Who has the highest salary?", context_user))

SELECT e.name 
FROM employees e 
JOIN salary s ON e.ID_usr = s.ID_usr 
ORDER BY s.salary DESC 
LIMIT 1;


In [35]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Who has the highest salary?", old_context_user))

This is your SQL:
```sql
SELECT e.name, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query selects the name and salary of the employee with the highest salary by joining the "employees" and "salary" tables on the ID_usr column. It then orders the results by salary in descending order and limits the output to only the top result, which represents the employee with the highest salary.


In [36]:
#new
print(return_CCRMSQL("Which institution has highest average salary?", context_user))

SELECT st.Institution, AVG(sa.salary) 
FROM studies st 
JOIN employees e ON st.ID_usr=e.ID_usr 
JOIN salary sa ON e.ID_usr=sa.ID_usr 
GROUP BY st.Institution 
ORDER BY AVG(sa.salary) DESC 
LIMIT 1;


In [37]:
#old
print(return_CCRMSQL("Which institution has highest average salary?" , old_context_user))

This is your SQL:
```sql
SELECT s.Institution, AVG(s.salary) AS avg_salary
FROM salary s
JOIN employees e ON s.ID_usr = e.ID_usr
GROUP BY s.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```

This SQL query retrieves the institution with the highest average salary by joining the `salary` table with the `employees` table on the `ID_usr` column. It calculates the average salary for each institution, orders the results in descending order of average salary, and then limits the output to only show the institution with the highest average salary.


In [38]:
#new
print(return_CCRMSQL("How many employees earn over 50000?", context_user))

```sql
SELECT COUNT(*) FROM salary WHERE salary > 50000;
```


In [40]:
#salary 
print(return_CCRMSQL("How many employees earn over 50000?", old_context_user))

This is your SQL:
```sql
SELECT COUNT(*) 
FROM employees 
JOIN salary ON employees.ID_usr = salary.ID_usr 
WHERE salary.salary > 50000;
```

This SQL query joins the "employees" and "salary" tables on the "ID_usr" column and then counts the number of employees whose salary is over 50000.


# COTECT_v2 - execice
 More examples + relationships + output format instruction

In [41]:
context_v2 = [ {'role':'system', 'content':"""
CREATE TABLE employees (
    ID_usr int primary key,
    name varchar(100)
);
-- Sample: (1,'John'), (2,'Maria'), (3,'David'), (4,'Sarah'), (5,'Mike')
CREATE TABLE salary (
    ID_usr int,
    year date,
    salary float,
    foreign key (ID_usr) references employees(ID_usr)
);
-- Sample: (1,'2023',55000), (2,'2023',62000), (3,'2023',48000), (4,'2023',71000)
CREATE TABLE studies (
    ID int,
    ID_usr int,
    educational_level int,
    Institution varchar(100),
    Years date,
    Speciality varchar(100),
    foreign key (ID_usr) references employees(ID_usr)
);
-- Sample: (1,1,3,'MIT','2015','CS'), (2,2,2,'Stanford','2016','Eng'), (3,3,1,'Harvard','2017','Math')
"""}]

In [42]:
context_v2.append( {'role':'system', 'content':"""
You are a SQL expert. Write simple, efficient SQLite queries.
-- Example 1 (Simple SELECT + JOIN):
Q: Who has highest salary?
SQL: SELECT e.name FROM employees e JOIN salary s ON e.ID_usr=s.ID_usr ORDER BY s.salary DESC LIMIT 1;
-- Example 2 (GROUP BY + AVG):
Q: Which institution pays highest avg?
SQL: SELECT st.Institution FROM studies st JOIN salary s ON st.ID_usr=s.ID_usr GROUP BY st.Institution ORDER BY AVG(s.salary) DESC LIMIT 1;
-- Example 3 (COUNT + WHERE):
Q: How many employees earn over 50000?
SQL: SELECT COUNT(*) FROM salary WHERE salary > 50000;
-- Example 4 (ORDER BY ascending):
Q: Who has lowest salary?
SQL: SELECT e.name FROM employees e JOIN salary s ON e.ID_usr=s.ID_usr ORDER BY s.salary ASC LIMIT 1;
Now answer: {question}
Provide ONLY the SQL query, no explanation.
"""})

In [46]:
print(return_CCRMSQL("what is the name of who has the higgest salary?", context_v2))
print("=" * 30 )
print(return_CCRMSQL("What is the biggest salary?", context_v2))
print("=" * 30 )
print(return_CCRMSQL("which institution pay less?", context_v2))
print("=" * 30 )
print(return_CCRMSQL("what is the name with the lowest salary?", context_v2))

SELECT e.name 
FROM employees e 
JOIN salary s ON e.ID_usr=s.ID_usr 
ORDER BY s.salary DESC 
LIMIT 1;
SELECT MAX(salary) FROM salary;
SELECT st.Institution 
FROM studies st 
JOIN salary s ON st.ID_usr=s.ID_usr 
GROUP BY st.Institution 
ORDER BY AVG(s.salary) ASC 
LIMIT 1;
SELECT e.name 
FROM employees e 
JOIN salary s ON e.ID_usr=s.ID_usr 
ORDER BY s.salary ASC 
LIMIT 1;


In [47]:
print(return_CCRMSQL("what is the name of who has the higgest salary?", old_context_user))
print("=" * 30 )
print(return_CCRMSQL("What is the biggest salary?", old_context_user))
print("=" * 30 )
print(return_CCRMSQL("which institution pay less?", old_context_user))
print("=" * 30 )
print(return_CCRMSQL("what is the name with the lowest salary?", old_context_user))

This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query selects the name of the employee with the highest salary by joining the "employees" table with the "salary" table on the ID_usr column, then ordering the results by salary in descending order and limiting the output to only the top result.
This is your SQL:
```sql
SELECT MAX(salary) AS max_salary FROM salary;
```

This SQL query selects the maximum salary from the "salary" table, returning the highest salary value.
This is your SQL:
```sql
SELECT Institution, MIN(salary) AS MinSalary
FROM employees
JOIN salary ON employees.ID_usr = salary.ID_usr
GROUP BY Institution
ORDER BY MinSalary;
```

This SQL query retrieves the institution that pays the lowest salary by joining the "employees" and "salary" tables on the ID_usr column, then calculates the minimum salary for each institution and orders the results by the minimum salary in ascending order

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

### Report lab-2

I founded some differences between the versions and promting: 

1. Create table + examples had cleared results than Json method 
2. More examples enhance the accuracy but cost more as longer promt 
3. Old prompt - Model added explanatory text ("This is your SQL...") 
4. Without sample rows - Model hallucinated columns that don't exist
- Example: Old prompt tried to use salary.Institution which doesn't exist (it's in studies table)
- Sample rows help the model understand which columns belong to which table
5. Experimented with Different Prompt Variations
I tested multiple ways of structuring the prompt to evaluate whether the model would produce consistent results and accurately understand the schema. This included:
- Varying the schema representation (JSON vs CREATE TABLE syntax)
- Adding/removing sample rows
- Testing different numbers of few-shot examples (2 vs 4 vs 8)
- Changing the output format instruction ("Provide only SQL" vs descriptive)

#### Conclusion
Following the Ohio University paper's recommendations (schema as code + sample rows + few-shot examples) significantly improved SQL generation accuracy compared to the basic JSON schema prompt.